# PPEM Application Profiling Vizualizer 

Author: K. Desnos & MAI-Code-1.1-Flash  
Version : 1.0 (Sep. 2026)

## 0. Profile the application
1. Make sure the `ENABLE_PROFILING` is enabled for your C project.   
   In VSCode, press `[F1]`, search for `CMake: Edit CMake Cache (UI)`, and set the `ENABLE_PROFILING` to true.
2. Clean, Build and Run the executable in debug or release config.
3. Run the executable, a `trace.out` file should be generated in the execution directory. 

## 1. Configuration & trace parsing
In the following cell, indicate the location of the trace file and the associated executable.
The given files will be parsed to extract profiling info.

In [5]:
trace_file = "../build/Release/trace.out"  # Replace with the actual path to your trace file
executable = "../build/Release/mandelbrot"  # Replace with the actual path to your executable

# Parse the given files
import trace_parser
calls, cores = trace_parser.parse_trace(trace_file, executable)

# Print number of function calls
print(f"Total function calls: {len(calls)}")

Total function calls: 462


## 2. Analyze profiling data

### 2.1. Call Tree
This first array displays the call tree extracted from the graph. Each function call appears in the hierarchy of function call, with its given statistics.

- `call_count`: number of times this function was executed.
- `total_cpu_time_including_children`: cumulative execution time for all calls of this function, including the time spent in callees.
- `total_cpu_time_excluding_children`: cumulative execution time spent directly in this function body, excluding callees.
- `avg_cpu_time_including_children`: average duration per call, including callee time. <!-- `total_cpu_time_including_children` / `call_count`.!-->
- `avg_cpu_time_excluding_children`: average duration per call, excluding callee time. <!-- `total_cpu_time_excluding_children` / `call_count`. -->
- `percent_of_parent_incl`: how much of the parent function’s inclusive time is due to this function. <!-- `total_cpu_time_including_children` / `parent_total_cpu_time_including_children` * 100. -->
- `percent_of_parent_excl`: how much of the parent function’s exclusive time is due to this function. <!-- `total_cpu_time_excluding_children` / `parent_total_cpu_time_including_children` * 100. -->
- `percent_of_total_incl`: percentage of the total CPU time attributed to this function, including the time spent in callees.
- `percent_of_total_excl`: percentage of the total CPU time attributed to this function, excluding the time spent in callees.


These values help identify which functions or call branches dominate total runtime and which parts are mainly overhead versus actual work.

In [7]:
# Build a call tree from the parsed calls
call_tree_roots = trace_parser.analyze_call_tree(calls)
call_tree_df = trace_parser.build_call_tree_dataframe(call_tree_roots)

# Choose the columns you want to display, and set the order here.
# Edit this list to keep only the metrics you care about.
selected_columns = [
    "call_count",
    "percent_of_total_incl",
    "percent_of_total_excl",
    "total_cpu_time_including_children",
    "total_cpu_time_excluding_children",
    "avg_cpu_time_including_children",
    "avg_cpu_time_excluding_children",
    "percent_of_parent_incl",
    "percent_of_parent_excl"
]

# Keep only the existing columns, in the desired order.
visible_columns = [col for col in selected_columns if col in call_tree_df.columns]
call_tree_df = call_tree_df[visible_columns]

call_tree_df

,call_count,percent_of_total_incl,percent_of_total_excl,total_cpu_time_including_children,total_cpu_time_excluding_children,avg_cpu_time_including_children,avg_cpu_time_excluding_children,percent_of_parent_incl,percent_of_parent_excl
display_name,,,,,,,,,
main,1,100.00,0.02,5024557,853,5024557.00,853.00,100.00,0.02
├── yuvDisplayInit,1,2.74,2.74,137508,137499,137508.00,137499.00,2.74,2.74
│ └── startTiming,4,0.00,0.00,9,9,2.25,2.25,0.01,0.01
├── startTiming,1,0.00,0.00,1,1,1.00,1.00,0.00,0.00
├── fillMandelbrot,39,89.08,89.08,4476008,4476008,114769.44,114769.44,89.08,89.08
├── MD5_Update,39,3.93,0.02,197511,1110,5064.38,28.46,3.93,0.02
│ ├── MD5_Init,39,0.00,0.00,29,29,0.74,0.74,0.01,0.01
│ ├── body,39,3.91,3.91,196285,196285,5032.95,5032.95,99.38,99.38
│ └── MD5_Final,39,0.00,0.00,87,60,2.23,1.54,0.04,0.03


### 2.1 Flame Graph view of the function call tree

This flame graph represents the inclusive CPU time of each function in the call tree. The wider a block, the more time it spent including its descendants.

In [ ]:
# Plot the call hierarchy as an inclusive-time flame graph
flame_graph = trace_parser.build_flamegraph(calls)
flame_graph

## 3 Per function statistics

### 3.1. Function statistics
This array gathers all statistics for each encountered function call such that a function calls encountered in several places in the profiles program are summed.

In [9]:
# Aggregate all occurrences of the same function across the whole profile.
# This sums the values of repeated calls encountered at different places in the program.
function_stats_df = trace_parser.build_function_statistics_dataframe(calls)

# Choose the columns you want to display, and set the order here.
# Edit this list to keep only the metrics you care about.
selected_columns = [
    "call_count",
    "percent_of_total_incl",
    "percent_of_total_excl",
    "total_cpu_time_including_children",
    "total_cpu_time_excluding_children",
    "avg_cpu_time_including_children",
    "avg_cpu_time_excluding_children",
    "function_file",
    "function_address"
]

# Keep only the existing columns, in the desired order.
visible_columns = [col for col in selected_columns if col in function_stats_df.columns]
function_stats_df = function_stats_df[visible_columns]

function_stats_df

,call_count,percent_of_total_incl,percent_of_total_excl,total_cpu_time_including_children,total_cpu_time_excluding_children,avg_cpu_time_including_children,avg_cpu_time_excluding_children,function_file,function_address
function_name,,,,,,,,,
main,1,100.00,0.02,5024557,853,5024557.00,853.00,/home/boris/Desktop/projet_ppem/src/main.c:20,0x402570
yuvDisplayInit,1,2.74,2.74,137508,137499,137508.00,137499.00,/home/boris/Desktop/projet_ppem/src/yuvDisplay...,0x404700
startTiming,44,0.00,0.00,40,40,0.91,0.91,/home/boris/Desktop/projet_ppem/src/clock.c:35,0x4028e0
fillMandelbrot,39,89.08,89.08,4476008,4476008,114769.44,114769.44,/home/boris/Desktop/projet_ppem/src/mandelbrot...,0x4029f0
MD5_Update,39,3.93,0.02,197511,1110,5064.38,28.46,/home/boris/Desktop/projet_ppem/src/md5.c:262,0x4040c0
MD5_Init,39,0.00,0.00,29,29,0.74,0.74,/home/boris/Desktop/projet_ppem/src/md5.c:212,0x403580
body,78,3.91,3.91,196312,196312,2516.82,2516.82,/home/boris/Desktop/projet_ppem/src/md5.c:105,0x4035d0
MD5_Final,39,0.00,0.00,87,60,2.23,1.54,/home/boris/Desktop/projet_ppem/src/md5.c:228,0x403db0
mandelbrotToYUV,39,3.55,3.55,178138,178138,4567.64,4567.64,/home/boris/Desktop/projet_ppem/src/mandelbrot...,0x402d20


### 3.2 Function execution time distribution

The following chart allows to visualize the distribution of execution time for each function. 

In [11]:
# Select a subset of functions to display in the box plot.
# Use the function names exactly as they appear in the trace/profile table.
# Set this to None to display all functions.
selected_boxplot_functions = [
    "thread_wrapper",
    "yuvDisplay",
    "body",
    "mandelbrotToYUV"
]

# Keep only the selected functions; if a name is missing from the trace, it is ignored.
box_plot = trace_parser.build_boxplot(calls, selected_boxplot_functions)
box_plot